In [ ]:
# Initialization
# Install dependencies
%pip install anthropic python-dotenv

In [2]:
# Environment load
# Load env variables and create API Client
from dotenv import load_dotenv

load_dotenv()

# Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5" #"claude-sonnet-5", claude-haiku-4-5

In [7]:
def add_user_message(messages, content):
    messages.append({"role": "user", "content": content})

def add_assistant_message(messages, content):
    messages.append({"role": "assistant", "content": content})


def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    message = client.messages.create(**params)

    return message.content[0].text

In [ ]:
messages = []

add_user_message(messages, "Generate a very short event bridge rule as a json")

chat(messages)
# we see the content below will add some opening markdown ticks, a json text, the json,
# and finally closing markdown

'```json\n{\n  "Name": "MySimpleRule",\n  "EventBusName": "default",\n  "EventPattern": {\n    "source": ["aws.ec2"],\n    "detail-type": ["EC2 Instance State-change Notification"],\n    "detail": {\n      "state": ["running"]\n    }\n  },\n  "State": "ENABLED",\n  "Targets": [\n    {\n      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:MyFunction",\n      "Id": "1"\n    }\n  ]\n}\n```'

In [ ]:
# this example will prefil the assistant message and add a terminating sequence, so we only get the raw json
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

chat(messages, stop_sequences=["```"])

#claude may add extra line chars or whitespace.. this is trivial to trim

'\n{\n  "Name": "MyRule",\n  "EventBusName": "default",\n  "EventPattern": {\n    "source": ["aws.ec2"],\n    "detail-type": ["EC2 Instance State-change Notification"],\n    "detail": {\n      "state": ["running"]\n    }\n  },\n  "State": "ENABLED",\n  "Targets": [\n    {\n      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:MyFunction",\n      "Id": "1"\n    }\n  ]\n}\n'

In [13]:
# prefill/stop sequence exercise
messages = []

add_user_message(messages, "Generate three different aws CLI commands. Each should be very short.")
add_assistant_message(messages, "Here are your commands in a single block--no explanations.  ```bash")

chat(messages, stop_sequences=["```"])

'\naws s3 ls\naws ec2 describe-instances\naws dynamodb list-tables\n'